# PTCG AI Agent — Observation Parser

**Pipeline Stage**: Stage 1 — Perception  
**Input**: Raw observation dict from the `cabt` engine  
**Output**: `ParsedObservation` (typed dataclass)  
**Reference**: [cabt API docs](https://matsuoinstitute.github.io/cabt/api.html)

---

## Why This Module Exists

The `cabt` engine delivers the game state as a raw Python dictionary. Before any strategic reasoning can happen, we need to convert this untyped blob into a clean, typed data structure. This is the **first layer** of the agent pipeline — it touches the environment API directly so that **no other module needs to**.

### Design Rationale

1. **Dataclasses over dicts** — downstream code gets autocomplete, type checking, and fails loudly on missing fields instead of silently returning `None` from `dict.get()`.
2. **Parse both `select_type` and `select_context`** — the heuristic engine needs both to decide what kind of decision is being made (e.g., "choose a card" during setup is very different from "choose a card" during discard).
3. **Keep a reference to the raw observation** — the game logger records the unprocessed data for offline training.

### How This Fits the Pipeline

```
Raw observation dict → [THIS MODULE] → ParsedObservation → State Encoder → Heuristic Engine → Action
```

This is the "eyes" of the agent. It *sees* the board, but it doesn't *understand* it yet. That's the state encoder's job (Stage 2).

## Imports

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
from enum import Enum, auto

---

## Section 1: Enums — Direct Mappings from the `cabt` API

These enums are **not invented**. They are direct mirrors of the `cabt` engine's own enum definitions (`api.SelectType`, `api.SelectContext`, `api.CardType`, etc.).

We re-define them here as pure Python enums so our agent code has **zero dependency** on the `cabt` library at import time. This matters because:

- **(a)** We can unit-test the agent without the `cabt` engine installed.
- **(b)** The submission bundles only our code; `cabt` is provided by Kaggle's runtime.

### `SelectType` — What kind of selection the engine is asking for

At each decision point, the engine tells us *what category* of choice we need to make. This determines which heuristic sub-routine activates downstream.

| Value | Meaning | Example |
|-------|---------|--------|
| `MAIN` | Main phase menu | Play card, attack, end turn |
| `CARD` | Choose a card | From hand, deck, or discard |
| `ATTACK` | Choose an attack | Which move to use |
| `EVOLVE` | Choose an evolution | Which Pokémon to evolve into |
| `YES_NO` | Binary decision | Use an ability? Retreat? |

In [ ]:
class SelectType(Enum):
    """What kind of selection the engine is asking for."""
    MAIN = "MAIN"                       # Main phase: play card, attack, end turn
    CARD = "CARD"                       # Choose a card (from hand, deck, discard)
    ATTACHED_CARD = "ATTACHED_CARD"     # Choose an attached card (energy, tool)
    CARD_OR_ATTACHED_CARD = "CARD_OR_ATTACHED_CARD"
    ENERGY = "ENERGY"                   # Choose an energy type
    SKILL = "SKILL"                     # Choose an ability to use
    ATTACK = "ATTACK"                   # Choose an attack
    EVOLVE = "EVOLVE"                   # Choose an evolution
    COUNT = "COUNT"                     # Choose a number (e.g., how many cards to draw)
    YES_NO = "YES_NO"                   # Binary decision
    SPECIAL_CONDITION = "SPECIAL_CONDITION"  # Choose a status condition

### `SelectContext` — The game situation in which the selection is being made

This is the **second axis** of decision classification. While `SelectType` tells us *what* we're choosing, `SelectContext` tells us *why*.

A `CARD` selection during `SETUP_ACTIVE_POKEMON` ("pick your starter") requires completely different strategic reasoning than a `CARD` selection during `DISCARD` ("which card to throw away"). The heuristic engine dispatches on the `(SelectType, SelectContext)` pair.

In [ ]:
class SelectContext(Enum):
    """The game situation in which the selection is being made."""
    MAIN = "MAIN"                           # Main phase action menu
    SETUP_ACTIVE_POKEMON = "SETUP_ACTIVE_POKEMON"
    SETUP_BENCH_POKEMON = "SETUP_BENCH_POKEMON"
    SWITCH = "SWITCH"                       # Forced or voluntary switch
    TO_ACTIVE = "TO_ACTIVE"
    TO_BENCH = "TO_BENCH"
    TO_FIELD = "TO_FIELD"
    TO_HAND = "TO_HAND"
    DISCARD = "DISCARD"
    TO_DECK = "TO_DECK"
    TO_DECK_BOTTOM = "TO_DECK_BOTTOM"
    TO_PRIZE = "TO_PRIZE"
    NOT_MOVE = "NOT_MOVE"
    DAMAGE_COUNTER = "DAMAGE_COUNTER"
    DAMAGE_COUNTER_ANY = "DAMAGE_COUNTER_ANY"
    DAMAGE = "DAMAGE"
    REMOVE_DAMAGE_COUNTER = "REMOVE_DAMAGE_COUNTER"
    HEAL = "HEAL"
    EVOLVES_FROM = "EVOLVES_FROM"
    EVOLVES_TO = "EVOLVES_TO"
    DEVOLVE = "DEVOLVE"
    ATTACH_FROM = "ATTACH_FROM"
    ATTACH_TO = "ATTACH_TO"
    DETACH_FROM = "DETACH_FROM"
    LOOK = "LOOK"
    EFFECT_TARGET = "EFFECT_TARGET"
    DISCARD_ENERGY_CARD = "DISCARD_ENERGY_CARD"
    DISCARD_TOOL_CARD = "DISCARD_TOOL_CARD"
    SWITCH_ENERGY_CARD = "SWITCH_ENERGY_CARD"
    DISCARD_CARD_OR_ATTACHED_CARD = "DISCARD_CARD_OR_ATTACHED_CARD"
    DISCARD_ENERGY = "DISCARD_ENERGY"
    TO_HAND_ENERGY = "TO_HAND_ENERGY"

### `CardType` — Types of cards in the PTCG

The card type determines which game rules apply. Notably:
- Only one **Supporter** can be played per turn
- **Tools** attach to Pokémon (one per Pokémon)
- **Stadium** cards persist on the field until replaced
- **Basic Energy** has unlimited plays per turn (one attach per turn to one Pokémon)

In [ ]:
class CardType(Enum):
    """Types of cards in the PTCG."""
    POKEMON = "POKEMON"
    ITEM = "ITEM"
    TOOL = "TOOL"
    SUPPORTER = "SUPPORTER"
    STADIUM = "STADIUM"
    BASIC_ENERGY = "BASIC_ENERGY"
    SPECIAL_ENERGY = "SPECIAL_ENERGY"

### `AreaType` — Board zones where cards can exist

Each card in the game occupies exactly one zone at any time. The transitions between zones (e.g., `HAND → ACTIVE`, `ACTIVE → DISCARD`) are the fundamental game actions.

In [ ]:
class AreaType(Enum):
    """Board areas where cards can exist."""
    DECK = "DECK"
    HAND = "HAND"
    DISCARD = "DISCARD"
    ACTIVE = "ACTIVE"
    BENCH = "BENCH"
    PRIZE = "PRIZE"
    STADIUM = "STADIUM"
    ENERGY = "ENERGY"
    TOOL = "TOOL"
    PRE_EVOLUTION = "PRE_EVOLUTION"
    PLAYER = "PLAYER"
    LOOKING = "LOOKING"

### `EnergyType` — Pokémon energy types

Energy types determine type matchups (weakness/resistance) and attack cost requirements. The `COLORLESS` type is satisfied by any energy, making it a wildcard in energy economy calculations.

In [ ]:
class EnergyType(Enum):
    """Pokémon energy types."""
    COLORLESS = "COLORLESS"
    GRASS = "GRASS"
    FIRE = "FIRE"
    WATER = "WATER"
    LIGHTNING = "LIGHTNING"
    PSYCHIC = "PSYCHIC"
    FIGHTING = "FIGHTING"
    DARKNESS = "DARKNESS"
    METAL = "METAL"
    DRAGON = "DRAGON"
    RAINBOW = "RAINBOW"
    TEAM_ROCKET = "TEAM_ROCKET"

---

## Section 2: Parsed Data Structures

These dataclasses define the **typed interface** between the perception layer (this module) and all downstream stages (state encoder, heuristic engine, value network, MCTS). 

Every downstream module operates on these structures — never on the raw observation dict. If the `cabt` API changes, only the parser functions in Section 3 need updating.

### `CardInfo` — A single card as seen in the observation

Captures all game-relevant properties of a single card. The `raw` field preserves the original dict for debugging and logging.

In [ ]:
@dataclass
class CardInfo:
    """A single card as seen in the observation."""
    card_id: str = ""
    card_name: str = ""
    card_type: str = ""  # We keep as string for flexibility; map to CardType when needed
    hp: int = 0
    max_hp: int = 0
    damage: int = 0
    energy_type: str = ""
    attached_energies: List[str] = field(default_factory=list)
    attached_tools: List[str] = field(default_factory=list)
    attacks: List[Dict[str, Any]] = field(default_factory=list)
    abilities: List[Dict[str, Any]] = field(default_factory=list)
    weakness: str = ""
    resistance: str = ""
    retreat_cost: int = 0
    status_conditions: List[str] = field(default_factory=list)
    stage: str = ""  # "BASIC", "STAGE1", "STAGE2", "MEGA", etc.
    evolves_from: str = ""
    pre_evolutions: List[str] = field(default_factory=list)
    raw: Dict[str, Any] = field(default_factory=dict)

### `PlayerState` — One player's complete board state

We separate `active` from `bench` because the heuristic engine needs to reason about them differently:
- **Active**: Can attack, can be attacked, can retreat
- **Bench**: Can receive energy, can evolve, is safe from most attacks

The `is_self` distinction (handled during parsing) means:
- **Our state**: `hand` contains full `CardInfo` objects with card identities
- **Opponent state**: `hand` may be empty; only `hand_count` is reliably available

This asymmetry encodes the **imperfect information** nature of PTCG directly into the data structure.

In [ ]:
@dataclass
class PlayerState:
    """
    One player's board state as parsed from the observation.
    """
    active: Optional[CardInfo] = None
    bench: List[CardInfo] = field(default_factory=list)
    hand: List[CardInfo] = field(default_factory=list)
    hand_count: int = 0  # Always available (even for opponent)
    deck_count: int = 0
    discard: List[CardInfo] = field(default_factory=list)
    prize_count: int = 6
    supporter_played: bool = False
    energy_attached_this_turn: bool = False
    raw: Dict[str, Any] = field(default_factory=dict)

### `LegalOption` — A single legal action the agent can take

The `cabt` engine presents a menu of options at each decision point. Each option has:
- An **index** (this is our return value — the integer we pass back to the engine)
- A **`select_type`** (what kind of choice: `MAIN`, `CARD`, `ATTACK`, etc.)
- A **`select_context`** (what game situation: `SETUP_ACTIVE_POKEMON`, `SWITCH`, etc.)
- Details about the specific card/action involved

> **Critical**: The engine guarantees all presented options are legal. We never need to validate legality — only rank quality.

In [ ]:
@dataclass
class LegalOption:
    """
    A single legal action the agent can take.
    """
    index: int = 0
    select_type: str = "MAIN"
    select_context: str = "MAIN"
    card_id: str = ""
    card_name: str = ""
    description: str = ""
    raw: Dict[str, Any] = field(default_factory=dict)

### `ParsedObservation` — The complete parsed game state at a single decision point

This is the **output of Stage 1 (Perception)**. Everything downstream — state encoder, heuristic engine, value network — operates on this structure, never on the raw dict.

#### Belief State Note

This structure encodes a **belief state** because it contains BOTH:
- **Observable information**: our hand, both discard piles, board positions
- **Uncertainty markers**: opponent `hand_count` without card identities, `deck_count` without ordering, prize cards hidden

The state encoder (Stage 2) will later convert these uncertainty markers into probabilistic estimates — e.g., computing `P(card_X ∈ opponent_hand)` from the pool of unseen cards.

In [ ]:
@dataclass
class ParsedObservation:
    """
    The complete parsed game state at a single decision point.
    """
    my_state: PlayerState = field(default_factory=PlayerState)
    opp_state: PlayerState = field(default_factory=PlayerState)
    legal_options: List[LegalOption] = field(default_factory=list)
    turn_number: int = 0
    game_phase: str = ""
    stadium_card: Optional[CardInfo] = None
    logs: List[str] = field(default_factory=list)
    raw_observation: Dict[str, Any] = field(default_factory=dict)

---

## Section 3: Parser Functions

These functions convert the raw `cabt` observation into the typed structures defined above. They are the **only code in the entire agent that touches the raw observation format**.

### Design principle: Defensive extraction

Every field access is wrapped in `_safe_get()` — a utility that handles `dict`, `object`, and `None` inputs without raising exceptions. This is **non-negotiable** for a competition agent: a `KeyError` during a live ladder match means an automatic loss.

### `_safe_get()` — Defensive field extraction

Handles three input types:
1. **`dict`** → uses `.get(key, default)`
2. **Object with attributes** → uses `getattr(obj, key, default)`
3. **`None`** → returns `default`

The `cabt` observation structure may change between engine versions, and different decision points may have different fields populated. We never want the agent to crash on a `KeyError` during a live match.

In [ ]:
def _safe_get(d: Any, key: str, default: Any = None) -> Any:
    """
    Safely extract a key from a dict-like object.
    """
    if isinstance(d, dict):
        return d.get(key, default)
    return getattr(d, key, default) if d is not None else default

### `_parse_card()` — Parse a single card from the observation

Extracts all game-relevant fields from a raw card dict/object. Handles multiple naming conventions (e.g., `'id'` vs `'card_id'`, `'hp'` vs `'current_hp'`) because the `cabt` engine uses different field names in different contexts (board vs hand vs discard).

Sub-structures (attacks, abilities, energies, tools, status conditions) are parsed into normalized Python lists for downstream consumption.

In [ ]:
def _parse_card(raw_card: Any) -> CardInfo:
    """Parse a single card from the observation into a CardInfo."""
    if raw_card is None:
        return CardInfo()
    
    card = CardInfo()
    card.raw = dict(raw_card) if isinstance(raw_card, dict) else {}
    
    # Core identity fields — handle multiple naming conventions
    card.card_id = str(_safe_get(raw_card, 'id', _safe_get(raw_card, 'card_id', '')))
    card.card_name = str(_safe_get(raw_card, 'name', _safe_get(raw_card, 'card_name', '')))
    card.card_type = str(_safe_get(raw_card, 'card_type', _safe_get(raw_card, 'type', '')))
    card.hp = int(_safe_get(raw_card, 'hp', _safe_get(raw_card, 'current_hp', 0)) or 0)
    card.max_hp = int(_safe_get(raw_card, 'max_hp', card.hp) or 0)
    card.damage = int(_safe_get(raw_card, 'damage', _safe_get(raw_card, 'damage_counters', 0)) or 0)
    card.energy_type = str(_safe_get(raw_card, 'energy_type', _safe_get(raw_card, 'pokemon_type', '')))
    card.retreat_cost = int(_safe_get(raw_card, 'retreat_cost', 0) or 0)
    card.stage = str(_safe_get(raw_card, 'stage', ''))
    card.evolves_from = str(_safe_get(raw_card, 'evolves_from', ''))
    card.weakness = str(_safe_get(raw_card, 'weakness', ''))
    card.resistance = str(_safe_get(raw_card, 'resistance', ''))
    
    # Parse attached energies (can be list of strings or dict of type:count)
    attached = _safe_get(raw_card, 'attached_energies', _safe_get(raw_card, 'energies', []))
    if isinstance(attached, list):
        card.attached_energies = [str(e) for e in attached]
    elif isinstance(attached, dict):
        card.attached_energies = [f"{k}:{v}" for k, v in attached.items()]
    
    # Parse attacks (cabt uses 'skills' in some contexts)
    attacks = _safe_get(raw_card, 'attacks', _safe_get(raw_card, 'skills', []))
    if isinstance(attacks, list):
        card.attacks = [dict(a) if isinstance(a, dict) else {'name': str(a)} for a in attacks]
    
    # Parse abilities
    abilities = _safe_get(raw_card, 'abilities', [])
    if isinstance(abilities, list):
        card.abilities = [dict(a) if isinstance(a, dict) else {'name': str(a)} for a in abilities]
    
    # Parse status conditions
    conditions = _safe_get(raw_card, 'special_conditions', _safe_get(raw_card, 'status', []))
    if isinstance(conditions, list):
        card.status_conditions = [str(c) for c in conditions]
    elif isinstance(conditions, str) and conditions:
        card.status_conditions = [conditions]
    
    # Parse tools
    tools = _safe_get(raw_card, 'attached_tools', _safe_get(raw_card, 'tools', []))
    if isinstance(tools, list):
        card.attached_tools = [str(t) for t in tools]
    
    return card

### `_parse_card_list()` — Parse a list of cards

Simple helper that applies `_parse_card()` to each element of a list. Returns an empty list for `None` or non-list inputs.

In [ ]:
def _parse_card_list(raw_list: Any) -> List[CardInfo]:
    """Parse a list of cards from the observation."""
    if not raw_list or not isinstance(raw_list, (list, tuple)):
        return []
    return [_parse_card(c) for c in raw_list]

### `_parse_player_state()` — Parse one player's board state

The `is_self` parameter controls **information asymmetry handling**:

| Field | `is_self=True` (our state) | `is_self=False` (opponent) |
|-------|---------------------------|---------------------------|
| `hand` | Full `CardInfo` list with card identities | May be empty list |
| `hand_count` | `len(hand)` | Integer from observation (always available) |
| `deck_count` | Integer (we can't see our own deck order either) | Integer |
| `discard` | Full `CardInfo` list | Full `CardInfo` list (discard is public) |

This asymmetry is **deliberate** — it encodes the imperfect-information nature of PTCG directly into the data structure. The state encoder later uses `hand_count` (for the opponent) and `hand` (for self) differently when computing belief-state features.

In [ ]:
def _parse_player_state(raw_player: Any, is_self: bool = True) -> PlayerState:
    """
    Parse a player's state from the observation.
    """
    if raw_player is None:
        return PlayerState()
    
    state = PlayerState()
    state.raw = dict(raw_player) if isinstance(raw_player, dict) else {}
    
    # Active Pokémon — may be a single object or a 1-element list
    active_raw = _safe_get(raw_player, 'active', _safe_get(raw_player, 'active_pokemon', None))
    if active_raw:
        if isinstance(active_raw, list) and len(active_raw) > 0:
            state.active = _parse_card(active_raw[0])
        else:
            state.active = _parse_card(active_raw)
    
    # Bench — always a list, up to 5 Pokémon
    bench_raw = _safe_get(raw_player, 'bench', _safe_get(raw_player, 'bench_pokemon', []))
    state.bench = _parse_card_list(bench_raw)
    
    # Hand — full card info for self, count-only for opponent
    hand_raw = _safe_get(raw_player, 'hand', [])
    if is_self:
        state.hand = _parse_card_list(hand_raw)
        state.hand_count = len(state.hand)
    else:
        # Opponent: we may only know the count
        if isinstance(hand_raw, list):
            state.hand = _parse_card_list(hand_raw)  # Might be empty
            state.hand_count = len(hand_raw)
        else:
            state.hand_count = int(hand_raw or 0)
    
    # Deck — count only (never card identities, even for self)
    deck_raw = _safe_get(raw_player, 'deck', _safe_get(raw_player, 'deck_count', 0))
    if isinstance(deck_raw, list):
        state.deck_count = len(deck_raw)
    else:
        state.deck_count = int(deck_raw or 0)
    
    # Discard pile — fully observable for both players
    discard_raw = _safe_get(raw_player, 'discard', _safe_get(raw_player, 'discard_pile', []))
    state.discard = _parse_card_list(discard_raw)
    
    # Prize cards — count only (face-down, hidden even from self)
    prize_raw = _safe_get(raw_player, 'prize', _safe_get(raw_player, 'prize_cards', 
                          _safe_get(raw_player, 'prize_count', 6)))
    if isinstance(prize_raw, list):
        state.prize_count = len(prize_raw)
    else:
        state.prize_count = int(prize_raw or 6)
    
    # Turn-level flags
    state.supporter_played = bool(_safe_get(raw_player, 'supporter_played', False))
    state.energy_attached_this_turn = bool(_safe_get(raw_player, 'energy_attached', False))
    
    return state

### `_parse_legal_options()` — Parse the legal action list

**Critical design decision**: The engine presents legal options as a list. We return the **INDEX** of our chosen option. This means the order matters — we preserve the original indexing exactly.

Handles two observation formats:
1. **Dict with select info**: `{select_type: "MAIN", select_context: "MAIN", options: [...]}`
2. **Flat list**: `[{id: ..., name: ...}, {id: ..., name: ...}, ...]`

The parser normalizes both formats into a uniform `List[LegalOption]`.

In [ ]:
def _parse_legal_options(raw_legal: Any) -> List[LegalOption]:
    """
    Parse the legal action list from the observation.
    """
    if not raw_legal:
        return []
    
    options = []
    
    # Format 1: dict with select_type/select_context wrapping the options
    if isinstance(raw_legal, dict):
        select_type = str(_safe_get(raw_legal, 'select_type', 'MAIN'))
        select_context = str(_safe_get(raw_legal, 'select_context', 'MAIN'))
        items = _safe_get(raw_legal, 'options', _safe_get(raw_legal, 'items', 
                          _safe_get(raw_legal, 'selects', [])))
        
        if isinstance(items, list):
            for i, item in enumerate(items):
                opt = LegalOption()
                opt.index = i
                opt.select_type = select_type
                opt.select_context = select_context
                if isinstance(item, dict):
                    opt.card_id = str(_safe_get(item, 'id', _safe_get(item, 'card_id', '')))
                    opt.card_name = str(_safe_get(item, 'name', _safe_get(item, 'card_name', '')))
                    opt.description = str(_safe_get(item, 'description', _safe_get(item, 'text', '')))
                    opt.raw = dict(item)
                else:
                    opt.description = str(item)
                options.append(opt)
    
    # Format 2: flat list of option objects/dicts
    elif isinstance(raw_legal, list):
        for i, item in enumerate(raw_legal):
            opt = LegalOption()
            opt.index = i
            if isinstance(item, dict):
                opt.select_type = str(_safe_get(item, 'select_type', 'MAIN'))
                opt.select_context = str(_safe_get(item, 'select_context', 'MAIN'))
                opt.card_id = str(_safe_get(item, 'id', _safe_get(item, 'card_id', '')))
                opt.card_name = str(_safe_get(item, 'name', _safe_get(item, 'card_name', '')))
                opt.description = str(_safe_get(item, 'description', _safe_get(item, 'text', '')))
                opt.raw = dict(item)
            else:
                opt.description = str(item)
            options.append(opt)
    
    return options

---

## Main Entry Point: `parse_observation()`

This is the **single entry point** for Stage 1 of the agent pipeline. It converts the raw observation from the `cabt` engine into a fully typed `ParsedObservation`.

### Key Implementation Details

1. **Player index resolution**: The engine tells us which player we are via `observation.player` (0 or 1). We use this to correctly assign `my_state` vs `opp_state`.

2. **`observation.current` unwrapping**: The observation may have a `current` key wrapping the actual state, or the state may be at the top level. We handle both.

3. **Fallback parsing**: If `player_states` is not a list, we try `self`/`opponent` keys as a fallback.

### Contract

- **Input**: Anything. `None`, a dict, an object with attributes — all handled.
- **Output**: A `ParsedObservation` with all fields populated (with sensible defaults for missing data).
- **Guarantee**: This function **never raises an exception**.

In [ ]:
def parse_observation(observation: Any) -> ParsedObservation:
    """
    MAIN ENTRY POINT — Stage 1 of the agent pipeline.
    
    Converts the raw observation dict from the cabt engine into a
    fully typed ParsedObservation.
    
    This function is the ONLY place in the entire agent that touches
    the raw observation format. If the cabt API changes, only this
    function needs to be updated.
    """
    if observation is None:
        return ParsedObservation()
    
    parsed = ParsedObservation()
    parsed.raw_observation = dict(observation) if isinstance(observation, dict) else {}
    
    # The observation may have a 'current' key wrapping the actual state
    current = _safe_get(observation, 'current', observation)
    
    # Parse player states
    # Convention: player_states[0] = us, player_states[1] = opponent
    # But this may vary — the engine tells us our player index
    player_index = int(_safe_get(observation, 'player', _safe_get(current, 'player', 0)) or 0)
    
    player_states = _safe_get(current, 'player_states', 
                              _safe_get(current, 'players', None))
    
    if player_states and isinstance(player_states, (list, tuple)) and len(player_states) >= 2:
        opp_index = 1 - player_index
        parsed.my_state = _parse_player_state(player_states[player_index], is_self=True)
        parsed.opp_state = _parse_player_state(player_states[opp_index], is_self=False)
    else:
        # Fallback: try 'self' and 'opponent' keys
        my_raw = _safe_get(current, 'self', _safe_get(current, 'player', None))
        opp_raw = _safe_get(current, 'opponent', _safe_get(current, 'opp', None))
        parsed.my_state = _parse_player_state(my_raw, is_self=True)
        parsed.opp_state = _parse_player_state(opp_raw, is_self=False)
    
    # Parse legal options
    legal_raw = _safe_get(current, 'legal', 
                          _safe_get(current, 'legal_actions',
                                    _safe_get(observation, 'legal', None)))
    parsed.legal_options = _parse_legal_options(legal_raw)
    
    # Game metadata
    parsed.turn_number = int(_safe_get(current, 'turn', 
                                       _safe_get(current, 'turn_number', 0)) or 0)
    parsed.game_phase = str(_safe_get(current, 'phase', 
                                      _safe_get(current, 'game_phase', '')) or '')
    
    # Stadium card in play (if any)
    stadium_raw = _safe_get(current, 'stadium', None)
    if stadium_raw:
        parsed.stadium_card = _parse_card(stadium_raw)
    
    # Game logs (text history of game events)
    logs_raw = _safe_get(current, 'logs', _safe_get(current, 'log', []))
    if isinstance(logs_raw, list):
        parsed.logs = [str(l) for l in logs_raw]
    elif isinstance(logs_raw, str):
        parsed.logs = [logs_raw]
    
    return parsed

---

## Smoke Test

Verify the parser works correctly with a synthetic observation that mirrors the `cabt` engine's output structure.

In [ ]:
# Synthetic observation mimicking cabt engine output
test_observation = {
    'player': 0,
    'current': {
        'player_states': [
            {  # Player 0 (us)
                'active': {
                    'id': 'sv8-001',
                    'name': 'Lucario ex',
                    'card_type': 'POKEMON',
                    'hp': 260,
                    'max_hp': 260,
                    'damage': 40,
                    'energy_type': 'FIGHTING',
                    'attached_energies': ['FIGHTING', 'FIGHTING', 'COLORLESS'],
                    'attacks': [
                        {'name': 'Aura Sphere', 'damage': 120, 'cost': ['FIGHTING', 'FIGHTING', 'COLORLESS']},
                        {'name': 'Rising Fist', 'damage': 160, 'cost': ['FIGHTING', 'FIGHTING', 'FIGHTING', 'COLORLESS']}
                    ],
                    'weakness': 'PSYCHIC',
                    'retreat_cost': 2,
                    'stage': 'STAGE1'
                },
                'bench': [
                    {'id': 'sv8-010', 'name': 'Riolu', 'hp': 70, 'max_hp': 70, 'energy_type': 'FIGHTING', 'stage': 'BASIC'},
                    {'id': 'sv8-015', 'name': 'Ralts', 'hp': 60, 'max_hp': 60, 'energy_type': 'PSYCHIC', 'stage': 'BASIC'}
                ],
                'hand': [
                    {'id': 'sv8-050', 'name': "Professor's Research", 'card_type': 'SUPPORTER'},
                    {'id': 'sv8-055', 'name': 'Nest Ball', 'card_type': 'ITEM'},
                    {'id': 'sv8-060', 'name': 'Fighting Energy', 'card_type': 'BASIC_ENERGY'}
                ],
                'deck': 40,
                'discard': [
                    {'id': 'sv8-051', 'name': 'Iono', 'card_type': 'SUPPORTER'}
                ],
                'prize_count': 5,
                'supporter_played': False,
                'energy_attached': False
            },
            {  # Player 1 (opponent)
                'active': {
                    'id': 'sv8-100', 
                    'name': 'Dragapult ex',
                    'hp': 320,
                    'max_hp': 320,
                    'damage': 0,
                    'energy_type': 'PSYCHIC',
                    'attached_energies': ['PSYCHIC', 'PSYCHIC'],
                    'weakness': 'DARKNESS',
                    'stage': 'STAGE2'
                },
                'bench': [
                    {'id': 'sv8-098', 'name': 'Dreepy', 'hp': 60, 'max_hp': 60, 'stage': 'BASIC'}
                ],
                'hand': 5,  # Opponent: count only, no card identities
                'deck': 35,
                'discard': [],
                'prize_count': 6
            }
        ],
        'legal': {
            'select_type': 'MAIN',
            'select_context': 'MAIN',
            'options': [
                {'id': 'play_supporter', 'name': "Professor's Research", 'description': 'Play Supporter'},
                {'id': 'play_item', 'name': 'Nest Ball', 'description': 'Play Item'},
                {'id': 'attach_energy', 'name': 'Fighting Energy', 'description': 'Attach Energy'},
                {'id': 'attack_0', 'name': 'Aura Sphere', 'description': 'Attack: 120 damage'},
                {'id': 'attack_1', 'name': 'Rising Fist', 'description': 'Attack: 160 damage'},
                {'id': 'end_turn', 'name': 'End Turn', 'description': 'Pass'}
            ]
        },
        'turn': 5,
        'phase': 'MAIN',
        'logs': ['Player 0 drew a card.', 'Player 1 attached Psychic Energy to Dragapult ex.']
    }
}

# Run the parser
parsed = parse_observation(test_observation)

print("=" * 60)
print("PARSE RESULT SUMMARY")
print("=" * 60)
print(f"\nTurn: {parsed.turn_number}")
print(f"Phase: {parsed.game_phase}")
print(f"\n--- My State ---")
print(f"  Active: {parsed.my_state.active.card_name} (HP: {parsed.my_state.active.hp}, Damage: {parsed.my_state.active.damage})")
print(f"  Energies: {parsed.my_state.active.attached_energies}")
print(f"  Bench: {[p.card_name for p in parsed.my_state.bench]}")
print(f"  Hand ({parsed.my_state.hand_count} cards): {[c.card_name for c in parsed.my_state.hand]}")
print(f"  Deck: {parsed.my_state.deck_count} cards")
print(f"  Discard: {[c.card_name for c in parsed.my_state.discard]}")
print(f"  Prizes remaining: {parsed.my_state.prize_count}")
print(f"  Supporter played: {parsed.my_state.supporter_played}")
print(f"  Energy attached: {parsed.my_state.energy_attached_this_turn}")
print(f"\n--- Opponent State ---")
print(f"  Active: {parsed.opp_state.active.card_name} (HP: {parsed.opp_state.active.hp})")
print(f"  Bench: {[p.card_name for p in parsed.opp_state.bench]}")
print(f"  Hand count: {parsed.opp_state.hand_count} (contents hidden)")
print(f"  Prizes remaining: {parsed.opp_state.prize_count}")
print(f"\n--- Legal Options ({len(parsed.legal_options)}) ---")
for opt in parsed.legal_options:
    print(f"  [{opt.index}] {opt.card_name}: {opt.description} (type={opt.select_type}, ctx={opt.select_context})")
print(f"\n--- Game Logs ---")
for log in parsed.logs:
    print(f"  > {log}")

---

## Edge Case Tests

The parser must **never crash**. These tests verify graceful handling of:
1. `None` observation
2. Empty observation dict
3. Missing fields
4. Flat list format for legal options
5. Opponent hand as integer (count only)

In [ ]:
# Test 1: None observation
result = parse_observation(None)
assert result.turn_number == 0
assert result.legal_options == []
assert result.my_state.active is None
print("✓ Test 1 passed: None observation handled gracefully")

# Test 2: Empty dict
result = parse_observation({})
assert result.turn_number == 0
assert result.legal_options == []
print("✓ Test 2 passed: Empty dict handled gracefully")

# Test 3: Missing fields
result = parse_observation({'current': {'turn': 3}})
assert result.turn_number == 3
assert result.my_state.hand_count == 0
print("✓ Test 3 passed: Missing fields use sensible defaults")

# Test 4: Flat list legal options
result = parse_observation({
    'current': {
        'legal': [
            {'select_type': 'ATTACK', 'select_context': 'MAIN', 'name': 'Tackle', 'description': '30 damage'},
            {'select_type': 'MAIN', 'select_context': 'MAIN', 'name': 'End Turn', 'description': 'Pass'}
        ]
    }
})
assert len(result.legal_options) == 2
assert result.legal_options[0].select_type == 'ATTACK'
assert result.legal_options[1].card_name == 'End Turn'
print("✓ Test 4 passed: Flat list legal options parsed correctly")

# Test 5: Opponent hand as integer
result = parse_observation({
    'current': {
        'player_states': [
            {'hand': [{'name': 'Card A'}, {'name': 'Card B'}]},
            {'hand': 7}  # Opponent: count only
        ]
    }
})
assert result.my_state.hand_count == 2
assert result.opp_state.hand_count == 7
assert len(result.opp_state.hand) == 0  # No card identities
print("✓ Test 5 passed: Opponent hand count parsed from integer")

# Test with full synthetic observation from above
result = parse_observation(test_observation)
assert result.my_state.active.card_name == 'Lucario ex'
assert result.my_state.active.hp == 260
assert result.my_state.active.damage == 40
assert len(result.my_state.active.attached_energies) == 3
assert len(result.my_state.bench) == 2
assert result.my_state.hand_count == 3
assert result.opp_state.active.card_name == 'Dragapult ex'
assert result.opp_state.hand_count == 5
assert len(result.legal_options) == 6
assert result.legal_options[3].card_name == 'Aura Sphere'
assert result.turn_number == 5
print("✓ Test 6 passed: Full synthetic observation parsed correctly")

print("\n" + "=" * 40)
print("ALL TESTS PASSED")
print("=" * 40)

---

## Next: Stage 2 — State Encoder

The `ParsedObservation` produced by this module feeds into the **State Encoder** (`state_encoder.py`), which projects it into the belief-state feature vector `φ(s) ∈ ℝ¹²⁸`. That feature vector then drives the heuristic engine, value network, and MCTS.

```
[THIS MODULE]          →  State Encoder  →  Heuristic Engine  →  Value Network  →  MCTS  →  action_index
ParsedObservation         φ(s) ∈ ℝ¹²⁸      ranked_options       V(s') ∈ [0,1]     N(a)      int
```